### Aula 11 - Unidades de medida inconsistentes

In [ ]:
# Rodar o codigo apenas uma vez para instalar as bibliotecas
# Depois comentar para evitar tentar instalar novamente em execucoes futuras
# !pip install pandas
# !pip install matplotlib

# Importacao das bibliotecas
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Carregando dataframe
df = pd.read_csv('aula_11_dataset.csv')

# Inspecao visual do dataframe
display(df)

In [ ]:
# O que sao unidades de medida inconsistentes?
# E quando a mesma coluna mistura unidades diferentes.
# Exemplo: 2m, 200cm e 2000mm representam o mesmo valor,
# mas sem padronizar o sistema interpreta como numeros diferentes.

# Em bases reais isso acontece por:
# - Sistemas diferentes de origem
# - Digitacao manual
# - Falta de regra de cadastro

df_unidades = df.copy()
df_unidades['Valor_Comprimento'] = df_unidades['Comprimento'].str.extract(r'([\d.]+)').astype(float)
df_unidades['Unidade_Comprimento'] = df_unidades['Comprimento'].str.extract(r'([a-zA-Z]+)$')
df_unidades['Valor_Peso'] = df_unidades['Peso'].str.extract(r'([\d.]+)').astype(float)
df_unidades['Unidade_Peso'] = df_unidades['Peso'].str.extract(r'([a-zA-Z]+)$')

display(df_unidades[['Lote', 'Produto', 'Comprimento', 'Unidade_Comprimento', 'Peso', 'Unidade_Peso']])
display(df_unidades['Unidade_Comprimento'].value_counts())
display(df_unidades['Unidade_Peso'].value_counts())

In [ ]:
# Impacto em relatorios, dashboards e medias (sem padronizacao)
# Aqui a media fica errada porque mistura m, cm e mm como se fosse a mesma escala.
medias_erradas = df_unidades.groupby('Produto').agg(
    Comprimento_Medio_Sem_Padronizacao=('Valor_Comprimento', 'mean'),
    Peso_Medio_Sem_Padronizacao=('Valor_Peso', 'mean')
).reset_index()

display(medias_erradas)

# Exemplo de filtro errado:
# o objetivo e encontrar itens com comprimento >= 5 metros.
# Sem padronizacao, usar >= 5 no valor bruto traz resultados incorretos.
filtro_errado_5m = df_unidades[df_unidades['Valor_Comprimento'] >= 5]
display(filtro_errado_5m[['Lote', 'Produto', 'Comprimento', 'Valor_Comprimento']])

In [ ]:
# Como padronizar unidades
# Neste exemplo vamos padronizar:
# - Comprimento para centimetros
# - Peso para quilogramas

def comprimento_para_cm(texto):
    valor = float(''.join(ch for ch in texto if ch.isdigit() or ch == '.'))
    unidade = ''.join(ch for ch in texto if ch.isalpha())

    if unidade == 'm':
        return valor * 100
    if unidade == 'cm':
        return valor
    if unidade == 'mm':
        return valor / 10
    return None

def peso_para_kg(texto):
    valor = float(''.join(ch for ch in texto if ch.isdigit() or ch == '.'))
    unidade = ''.join(ch for ch in texto if ch.isalpha())

    if unidade == 'kg':
        return valor
    if unidade == 'g':
        return valor / 1000
    return None

df_padronizado = df.copy()
df_padronizado['Comprimento_cm'] = df_padronizado['Comprimento'].apply(comprimento_para_cm)
df_padronizado['Peso_kg'] = df_padronizado['Peso'].apply(peso_para_kg)

display(df_padronizado[['Lote', 'Produto', 'Comprimento', 'Comprimento_cm', 'Peso', 'Peso_kg']])

In [ ]:
# Comparacao antes/depois da padronizacao
medias_corretas = df_padronizado.groupby('Produto').agg(
    Comprimento_Medio_cm=('Comprimento_cm', 'mean'),
    Peso_Medio_kg=('Peso_kg', 'mean')
).reset_index()

comparacao_medias = medias_erradas.merge(medias_corretas, on='Produto', how='left')
display(comparacao_medias)

# Filtro correto para itens com comprimento >= 5 metros
# 5 metros = 500 centimetros
filtro_correto_5m = df_padronizado[df_padronizado['Comprimento_cm'] >= 500]
display(filtro_correto_5m[['Lote', 'Produto', 'Comprimento', 'Comprimento_cm']])

In [ ]:
# Impacto em modelos preditivos
# Features sem padronizacao geram coeficientes inconsistentes e previsoes ruins.
# Depois da padronizacao, as colunas ficam comparaveis na mesma unidade.

# Visual simples para comparar a escala antes/depois
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_unidades['Valor_Comprimento'].plot(kind='hist', bins=8, ax=axes[0], title='Sem padronizacao')
df_padronizado['Comprimento_cm'].plot(kind='hist', bins=8, ax=axes[1], title='Padronizado em cm')
plt.tight_layout()
plt.show()

In [ ]:
# Boas praticas para evitar unidades inconsistentes
# 1) Definir unidade padrao por coluna (ex: altura_cm, peso_kg)
# 2) Documentar essa unidade no dicionario de dados
# 3) Validar entrada de dados no sistema
# 4) Separar valor e unidade quando necessario
# 5) Usar nomes de coluna claros

# Exemplo final com colunas prontas para analise
df_final = df_padronizado[['Lote', 'Produto', 'Material', 'Comprimento_cm', 'Peso_kg']].copy()
display(df_final)

# Exemplo alternativo com valor e unidade separados
df_separado = df_unidades[['Lote', 'Produto', 'Valor_Comprimento', 'Unidade_Comprimento', 'Valor_Peso', 'Unidade_Peso']].copy()
display(df_separado)